## Dataset Preparation & Preprocessing


In [1]:
# required libraries
import os
import numpy as np
import pandas as pd

import data.processor as preprocess
# Split into train/test sets
from sklearn.model_selection import train_test_split

raw_data_path = os.path.join(os.getcwd(), 'data/datasets')
processed_data_path = os.path.join(os.getcwd(), 'data/preprocessed')
# Create directory if it doesn't exist
os.makedirs(processed_data_path, exist_ok=True)

In [2]:
# ignore future warnings by pytorch
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [3]:
# load dataset configuration from config file in data folder (yaml)
CONFIG = preprocess.read_yaml(os.path.join('data', 'config.yaml'))

### Adult Income Dataset

In [4]:
config = CONFIG['dataset'][0]

In [5]:
adult_preprocessor = preprocess.DatasetPreprocessor(numerical_cols=config['columns']['numerical_cols'] or [],
                                                    ordinal_cols=config['columns']['ordinal_cols'] or [],
                                                    binary_cols=config['columns']['binary_cols'] or [],
                                                    categorical_cols=config['columns']['categorical_cols'] or [],
                                                    target_col=config['columns']['target_col'])



In [6]:
# Load Adult dataset
adult_df = pd.read_csv(config['path'], header=None)
adult_df = adult_df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
adult_df.columns = config['columns']['all_cols']

In [7]:
# Preprocess dataset
preprocessed_df, preprocessed_oh_df, categories = adult_preprocessor.preprocess(adult_df)
print(f"Preprocessed DataFrame shape: {preprocessed_df.shape}")
print(f"Preprocessed One-Hot Encoded DataFrame shape: {preprocessed_oh_df.shape}")
print(f"Categories: {categories}")

Dropped columns with more than 50.0% missing values: ['education_num', 'fnlwgt']
Preprocessed DataFrame shape: (30162, 13)
Preprocessed One-Hot Encoded DataFrame shape: (30162, 102)
Categories: [16, 7, 7, 14, 6, 5, 41]


In [8]:
# Convert to tensors
X, y = adult_preprocessor.to_tensor(preprocessed_df, config['columns']['target_col'])
X_oh, y_oh = adult_preprocessor.to_tensor(preprocessed_oh_df, config['columns']['target_col'])

In [9]:

# First split into train+val (80%) and test (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Then split train+val into train (87.5% of 80% = 70%) and val (12.5% of 80% = 10%)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.125, random_state=42)

# First split into train+val (80%) and test (20%)
X_oh_train_val, X_oh_test, y_oh_train_val, y_oh_test = train_test_split(X_oh, y_oh, test_size=0.2, random_state=42)
# Then split train+val into train (87.5% of 80% = 70%) and val (12.5% of 80% = 10%)
X_oh_train, X_oh_val, y_oh_train, y_oh_val = train_test_split(X_oh_train_val, y_oh_train_val, test_size=0.125, random_state=42)


In [10]:
# Save preprocessor state and processed data
adult_preprocessor.save_preprocessor(save_dir=os.path.join(processed_data_path, 'adult'))
adult_preprocessor.save_data(X_train, X_val, X_test, y_train, y_val, y_test,
                             save_dir=os.path.join(processed_data_path, 'adult'))
# Save preprocessor state and processed data
adult_preprocessor.save_preprocessor(save_dir=os.path.join(processed_data_path, 'adult_oh'))
adult_preprocessor.save_data(X_oh_train, X_oh_val, X_oh_test, y_oh_train, y_oh_val, y_oh_test,
                             save_dir=os.path.join(processed_data_path, 'adult_oh'))

In [11]:
# Verify saved data
loaded_X_train, loaded_X_val, loaded_X_test, loaded_y_train, loaded_y_val, loaded_y_test = \
    adult_preprocessor.load_data(save_dir=os.path.join(processed_data_path, 'adult'))
print(f"Adult - Loaded data shapes:")
print(f"X_train: {loaded_X_train.shape}, y_train: {loaded_y_train.shape}")
print(f"X_test: {loaded_X_test.shape}, y_test: {loaded_y_test.shape}")

Adult - Loaded data shapes:
X_train: torch.Size([21112, 12]), y_train: torch.Size([21112])
X_test: torch.Size([6033, 12]), y_test: torch.Size([6033])


### Phishing URL Dataset

In [12]:
config = CONFIG['dataset'][1]

In [13]:
url_preprocessor = preprocess.DatasetPreprocessor(numerical_cols=config['columns']['numerical_cols'] or [],
                                                    ordinal_cols=config['columns']['ordinal_cols'] or [],
                                                    binary_cols=config['columns']['binary_cols'] or [],
                                                    categorical_cols=config['columns']['categorical_cols'] or [],
                                                    target_col=config['columns']['target_col'])


In [14]:
# Load url dataset
url_df = pd.read_csv(config['path'])
url_df.columns = config['columns']['all_cols']


In [15]:
# Preprocess dataset
preprocessed_df, preprocessed_oh_df, categories = url_preprocessor.preprocess(url_df)
print(f"Preprocessed DataFrame shape: {preprocessed_df.shape}")
print(f"Preprocessed One-Hot Encoded DataFrame shape: {preprocessed_oh_df.shape}")
print(f"Categories for embedding: {categories}")

Dropped columns with more than 50.0% missing values: ['port', 'url']
Preprocessed DataFrame shape: (11430, 87)
Preprocessed One-Hot Encoded DataFrame shape: (11430, 89)
Categories for embedding: [3]


In [16]:
# Convert to tensors
X, y = url_preprocessor.to_tensor(preprocessed_df, config['columns']['target_col'])
X_oh, y_oh = url_preprocessor.to_tensor(preprocessed_oh_df, config['columns']['target_col'])

In [17]:
# Split into train/test sets
from sklearn.model_selection import train_test_split
# First split into train+val (80%) and test (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Then split train+val into train (87.5% of 80% = 70%) and val (12.5% of 80% = 10%)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.125, random_state=42)

# First split into train+val (80%) and test (20%)
X_oh_train_val, X_oh_test, y_oh_train_val, y_oh_test = train_test_split(X_oh, y_oh, test_size=0.2, random_state=42)
# Then split train+val into train (87.5% of 80% = 70%) and val (12.5% of 80% = 10%)
X_oh_train, X_oh_val, y_oh_train, y_oh_val = train_test_split(X_oh_train_val, y_oh_train_val, test_size=0.125, random_state=42)

In [18]:
# Save preprocessor state and processed data
url_preprocessor.save_preprocessor(save_dir=os.path.join(processed_data_path, 'phishing_url'))
url_preprocessor.save_data(X_train, X_val, X_test, y_train, y_val, y_test,
                             save_dir=os.path.join(processed_data_path, 'phishing_url'))
# Save preprocessor state and processed data
url_preprocessor.save_preprocessor(save_dir=os.path.join(processed_data_path, 'phishing_url_oh'))
url_preprocessor.save_data(X_oh_train, X_oh_val, X_oh_test, y_oh_train, y_oh_val, y_oh_test,
                             save_dir=os.path.join(processed_data_path, 'phishing_url_oh'))

In [19]:
# Verify saved data
loaded_X_train, loaded_X_val, loaded_X_test, loaded_y_train, loaded_y_val, loaded_y_test = \
    url_preprocessor.load_data(save_dir=os.path.join(processed_data_path, 'phishing_url'))
print(f"phishing_url - Loaded data shapes:")
print(f"X_train: {loaded_X_train.shape}, y_train: {loaded_y_train.shape}")
print(f"X_test: {loaded_X_test.shape}, y_test: {loaded_y_test.shape}")

phishing_url - Loaded data shapes:
X_train: torch.Size([8001, 86]), y_train: torch.Size([8001])
X_test: torch.Size([2286, 86]), y_test: torch.Size([2286])


### PenDigits Dataset

In [20]:
config = CONFIG['dataset'][3]

In [21]:
pendigit_preprocessor = preprocess.DatasetPreprocessor(numerical_cols=config['columns']['numerical_cols'] or [],
                                                    ordinal_cols=config['columns']['ordinal_cols'] or [],
                                                    binary_cols=config['columns']['binary_cols'] or [],
                                                    categorical_cols=config['columns']['categorical_cols'] or [],
                                                    target_col=config['columns']['target_col'])

In [22]:
# Load PenDigit dataset
pendigit_df = pd.read_csv(config['path'])
pendigit_df.columns = config['columns']['all_cols']

In [23]:
# Preprocess dataset
preprocessed_df, preprocessed_oh_df, categories = pendigit_preprocessor.preprocess(pendigit_df)
print(f"Preprocessed DataFrame shape: {preprocessed_df.shape}")
print(f"Preprocessed One-Hot Encoded DataFrame shape: {preprocessed_oh_df.shape}")
print(f"Categories for embedding: {categories}")


Dropped columns with more than 50.0% missing values: ['id']
Preprocessed DataFrame shape: (10992, 17)
Preprocessed One-Hot Encoded DataFrame shape: (10992, 17)
Categories for embedding: []


In [24]:
# Convert to tensors
X, y = pendigit_preprocessor.to_tensor(preprocessed_df, config['columns']['target_col'])
X_oh, y_oh = pendigit_preprocessor.to_tensor(preprocessed_oh_df, config['columns']['target_col'])

In [25]:
# Split into train/test sets
from sklearn.model_selection import train_test_split
# First split into train+val (80%) and test (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Then split train+val into train (87.5% of 80% = 70%) and val (12.5% of 80% = 10%)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.125, random_state=42)

# First split into train+val (80%) and test (20%)
X_oh_train_val, X_oh_test, y_oh_train_val, y_oh_test = train_test_split(X_oh, y_oh, test_size=0.2, random_state=42)
# Then split train+val into train (87.5% of 80% = 70%) and val (12.5% of 80% = 10%)
X_oh_train, X_oh_val, y_oh_train, y_oh_val = train_test_split(X_oh_train_val, y_oh_train_val, test_size=0.125, random_state=42)

In [26]:
# Save preprocessor state and processed data
pendigit_preprocessor.save_preprocessor(save_dir=os.path.join(processed_data_path, 'pendigit'))
pendigit_preprocessor.save_data(X_train, X_val, X_test, y_train, y_val, y_test,
                             save_dir=os.path.join(processed_data_path, 'pendigit'))
# Save preprocessor state and processed data
pendigit_preprocessor.save_preprocessor(save_dir=os.path.join(processed_data_path, 'pendigit_oh'))
pendigit_preprocessor.save_data(X_oh_train, X_oh_val, X_oh_test, y_oh_train, y_oh_val, y_oh_test,
                             save_dir=os.path.join(processed_data_path, 'pendigit_oh'))

In [27]:
# Verify saved data
loaded_X_train, loaded_X_val, loaded_X_test, loaded_y_train, loaded_y_val, loaded_y_test = \
    pendigit_preprocessor.load_data(save_dir=os.path.join(processed_data_path, 'pendigit'))
print(f"PenDigit - Loaded data shapes:")
print(f"X_train: {loaded_X_train.shape}, y_train: {loaded_y_train.shape}")
print(f"X_test: {loaded_X_test.shape}, y_test: {loaded_y_test.shape}")

PenDigit - Loaded data shapes:
X_train: torch.Size([7693, 16]), y_train: torch.Size([7693])
X_test: torch.Size([2199, 16]), y_test: torch.Size([2199])


### Mushrooms Dataset

In [28]:
config = CONFIG['dataset'][4]

In [29]:
mushroom_preprocessor = preprocess.DatasetPreprocessor(numerical_cols=config['columns']['numerical_cols'] or [],
                                                    ordinal_cols=config['columns']['ordinal_cols'] or [],
                                                    binary_cols=config['columns']['binary_cols'] or [],
                                                    categorical_cols=config['columns']['categorical_cols'] or [],
                                                    target_col=config['columns']['target_col'])

In [30]:
# Load Mushroom dataset
mushroom_df = pd.read_csv(config['path'])
mushroom_df.columns = config['columns']['all_cols']

In [31]:
# Preprocess dataset
preprocessed_df, preprocessed_oh_df, categories = mushroom_preprocessor.preprocess(mushroom_df)
print(f"Preprocessed DataFrame shape: {preprocessed_df.shape}")
print(f"Preprocessed One-Hot Encoded DataFrame shape: {preprocessed_oh_df.shape}")
print(f"Categories for embedding: {categories}")


Dropped columns with more than 50.0% missing values: []
Preprocessed DataFrame shape: (5644, 23)
Preprocessed One-Hot Encoded DataFrame shape: (5644, 96)
Categories for embedding: [6, 4, 8, 7, 2, 2, 9, 4, 4, 4, 7, 7, 2, 3, 4, 6, 6, 6]


In [32]:
# Convert to tensors
X, y = mushroom_preprocessor.to_tensor(preprocessed_df, config['columns']['target_col'])
X_oh, y_oh = mushroom_preprocessor.to_tensor(preprocessed_oh_df, config['columns']['target_col'])

In [33]:
# Split into train/test sets
from sklearn.model_selection import train_test_split
# First split into train+val (80%) and test (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Then split train+val into train (87.5% of 80% = 70%) and val (12.5% of 80% = 10%)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.125, random_state=42)

# First split into train+val (80%) and test (20%)
X_oh_train_val, X_oh_test, y_oh_train_val, y_oh_test = train_test_split(X_oh, y_oh, test_size=0.2, random_state=42)
# Then split train+val into train (87.5% of 80% = 70%) and val (12.5% of 80% = 10%)
X_oh_train, X_oh_val, y_oh_train, y_oh_val = train_test_split(X_oh_train_val, y_oh_train_val, test_size=0.125, random_state=42)

In [34]:
# Save preprocessor state and processed data
mushroom_preprocessor.save_preprocessor(save_dir=os.path.join(processed_data_path, 'mushroom'))
mushroom_preprocessor.save_data(X_train, X_val, X_test, y_train, y_val, y_test,
                             save_dir=os.path.join(processed_data_path, 'mushroom'))
# Save preprocessor state and processed data
mushroom_preprocessor.save_preprocessor(save_dir=os.path.join(processed_data_path, 'mushroom_oh'))
mushroom_preprocessor.save_data(X_oh_train, X_oh_val, X_oh_test, y_oh_train, y_oh_val, y_oh_test,
                             save_dir=os.path.join(processed_data_path, 'mushroom_oh'))

In [35]:
# Verify saved data
loaded_X_train, loaded_X_val, loaded_X_test, loaded_y_train, loaded_y_val, loaded_y_test = \
    mushroom_preprocessor.load_data(save_dir=os.path.join(processed_data_path, 'mushroom'))
print(f"mushroom - Loaded data shapes:")
print(f"X_train: {loaded_X_train.shape}, y_train: {loaded_y_train.shape}")
print(f"X_test: {loaded_X_test.shape}, y_test: {loaded_y_test.shape}")

mushroom - Loaded data shapes:
X_train: torch.Size([3950, 22]), y_train: torch.Size([3950])
X_test: torch.Size([1129, 22]), y_test: torch.Size([1129])


### MiniBooNE Dataset

In [36]:
config = CONFIG['dataset'][6]

In [37]:
MiniBooNE_preprocessor = preprocess.DatasetPreprocessor(numerical_cols=config['columns']['numerical_cols'] or [],
                                                    ordinal_cols=config['columns']['ordinal_cols'] or [],
                                                    binary_cols=config['columns']['binary_cols'] or [],
                                                    categorical_cols=config['columns']['categorical_cols'] or [],
                                                    target_col=config['columns']['target_col'])

In [38]:
# Load MiniBooNE dataset
from scipy.io import arff
MiniBooNE_df = arff.loadarff(config['path'])
MiniBooNE_df = pd.DataFrame(MiniBooNE_df[0])
MiniBooNE_df.columns = config['columns']['all_cols']

In [39]:
# Preprocess dataset
preprocessed_df, preprocessed_oh_df, categories = MiniBooNE_preprocessor.preprocess(MiniBooNE_df)
print(f"Preprocessed DataFrame shape: {preprocessed_df.shape}")
print(f"Preprocessed One-Hot Encoded DataFrame shape: {preprocessed_oh_df.shape}")
print(f"Categories for embedding: {categories}")



Dropped columns with more than 50.0% missing values: []
Preprocessed DataFrame shape: (130064, 51)
Preprocessed One-Hot Encoded DataFrame shape: (130064, 51)
Categories for embedding: []


In [40]:
# Convert to tensors
X, y = MiniBooNE_preprocessor.to_tensor(preprocessed_df, config['columns']['target_col'])
X_oh, y_oh = MiniBooNE_preprocessor.to_tensor(preprocessed_oh_df, config['columns']['target_col'])

In [41]:
# Split into train/test sets
from sklearn.model_selection import train_test_split
# First split into train+val (80%) and test (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Then split train+val into train (87.5% of 80% = 70%) and val (12.5% of 80% = 10%)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.125, random_state=42)

# First split into train+val (80%) and test (20%)
X_oh_train_val, X_oh_test, y_oh_train_val, y_oh_test = train_test_split(X_oh, y_oh, test_size=0.2, random_state=42)
# Then split train+val into train (87.5% of 80% = 70%) and val (12.5% of 80% = 10%)
X_oh_train, X_oh_val, y_oh_train, y_oh_val = train_test_split(X_oh_train_val, y_oh_train_val, test_size=0.125, random_state=42)

In [42]:
# Save preprocessor state and processed data
MiniBooNE_preprocessor.save_preprocessor(save_dir=os.path.join(processed_data_path, 'MiniBooNE'))
MiniBooNE_preprocessor.save_data(X_train, X_val, X_test, y_train, y_val, y_test,
                             save_dir=os.path.join(processed_data_path, 'MiniBooNE'))
# Save preprocessor state and processed data
MiniBooNE_preprocessor.save_preprocessor(save_dir=os.path.join(processed_data_path, 'MiniBooNE_oh'))
MiniBooNE_preprocessor.save_data(X_oh_train, X_oh_val, X_oh_test, y_oh_train, y_oh_val, y_oh_test,
                             save_dir=os.path.join(processed_data_path, 'MiniBooNE_oh'))

In [43]:
# Verify saved data
loaded_X_train, loaded_X_val, loaded_X_test, loaded_y_train, loaded_y_val, loaded_y_test = \
    MiniBooNE_preprocessor.load_data(save_dir=os.path.join(processed_data_path, 'MiniBooNE'))
print(f"MiniBooNE - Loaded data shapes:")
print(f"X_train: {loaded_X_train.shape}, y_train: {loaded_y_train.shape}")
print(f"X_test: {loaded_X_test.shape}, y_test: {loaded_y_test.shape}")

MiniBooNE - Loaded data shapes:
X_train: torch.Size([91044, 50]), y_train: torch.Size([91044])
X_test: torch.Size([26013, 50]), y_test: torch.Size([26013])


### german_credit Dataset

In [44]:
config = CONFIG['dataset'][7]

In [45]:
german_credit_preprocessor = preprocess.DatasetPreprocessor(numerical_cols=config['columns']['numerical_cols'] or [],
                                                    ordinal_cols=config['columns']['ordinal_cols'] or [],
                                                    binary_cols=config['columns']['binary_cols'] or [],
                                                    categorical_cols=config['columns']['categorical_cols'] or [],
                                                    target_col=config['columns']['target_col'])


In [46]:
# Load german_credit dataset
from scipy.io import arff
german_credit_df = arff.loadarff(config['path'])
german_credit_df = pd.DataFrame(german_credit_df[0])
german_credit_df.columns = config['columns']['all_cols']


In [ ]:
# Preprocess dataset
preprocessed_df, preprocessed_oh_df, categories = german_credit_preprocessor.preprocess(german_credit_df)
print(f"Preprocessed DataFrame shape: {preprocessed_df.shape}")
print(f"Preprocessed One-Hot Encoded DataFrame shape: {preprocessed_oh_df.shape}")
print(f"Categories for embedding: {categories}")


Dropped columns with more than 50.0% missing values: ['existing_credits']
Preprocessed DataFrame shape: (1000, 20)
Preprocessed One-Hot Encoded DataFrame shape: (1000, 59)
Categories for embedding: [4, 5, 10, 5, 5, 4, 3, 4, 3, 3, 4]


In [48]:
# Convert to tensors
X, y = german_credit_preprocessor.to_tensor(preprocessed_df, config['columns']['target_col'])
X_oh, y_oh = german_credit_preprocessor.to_tensor(preprocessed_oh_df, config['columns']['target_col'])

In [49]:
# Split into train/test sets
from sklearn.model_selection import train_test_split
# First split into train+val (80%) and test (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Then split train+val into train (87.5% of 80% = 70%) and val (12.5% of 80% = 10%)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.125, random_state=42)

# First split into train+val (80%) and test (20%)
X_oh_train_val, X_oh_test, y_oh_train_val, y_oh_test = train_test_split(X_oh, y_oh, test_size=0.2, random_state=42)
# Then split train+val into train (87.5% of 80% = 70%) and val (12.5% of 80% = 10%)
X_oh_train, X_oh_val, y_oh_train, y_oh_val = train_test_split(X_oh_train_val, y_oh_train_val, test_size=0.125, random_state=42)


In [50]:
# Save preprocessor state and processed data
german_credit_preprocessor.save_preprocessor(save_dir=os.path.join(processed_data_path, 'german_credit'))
german_credit_preprocessor.save_data(X_train, X_val, X_test, y_train, y_val, y_test,
                             save_dir=os.path.join(processed_data_path, 'german_credit'))
# Save preprocessor state and processed data
german_credit_preprocessor.save_preprocessor(save_dir=os.path.join(processed_data_path, 'german_credit_oh'))
german_credit_preprocessor.save_data(X_oh_train, X_oh_val, X_oh_test, y_oh_train, y_oh_val, y_oh_test,
                             save_dir=os.path.join(processed_data_path, 'german_credit_oh'))


In [51]:
# Verify saved data
loaded_X_train, loaded_X_val, loaded_X_test, loaded_y_train, loaded_y_val, loaded_y_test = \
    german_credit_preprocessor.load_data(save_dir=os.path.join(processed_data_path, 'german_credit'))
print(f"german_credit - Loaded data shapes:")
print(f"X_train: {loaded_X_train.shape}, y_train: {loaded_y_train.shape}")

german_credit - Loaded data shapes:
X_train: torch.Size([700, 19]), y_train: torch.Size([700])


### electricity Dataset

In [52]:
config = CONFIG['dataset'][8]

In [53]:
electricity_preprocessor = preprocess.DatasetPreprocessor(numerical_cols=config['columns']['numerical_cols'] or [],
                                                    ordinal_cols=config['columns']['ordinal_cols'] or [],
                                                    binary_cols=config['columns']['binary_cols'] or [],
                                                    categorical_cols=config['columns']['categorical_cols'] or [],
                                                    target_col=config['columns']['target_col'])


In [54]:
# Load german_credit dataset
from scipy.io import arff
electricity_df = arff.loadarff(config['path'])
electricity_df = pd.DataFrame(electricity_df[0])
electricity_df.columns = config['columns']['all_cols']


In [55]:
# Preprocess dataset
preprocessed_df, preprocessed_oh_df, categories = electricity_preprocessor.preprocess(electricity_df)
print(f"Preprocessed DataFrame shape: {preprocessed_df.shape}")
print(f"Preprocessed One-Hot Encoded DataFrame shape: {preprocessed_oh_df.shape}")
print(f"Categories for embedding: {categories}")


Dropped columns with more than 50.0% missing values: []
Preprocessed DataFrame shape: (45312, 9)
Preprocessed One-Hot Encoded DataFrame shape: (45312, 15)
Categories for embedding: [7]


In [56]:
# Convert to tensors
X, y = electricity_preprocessor.to_tensor(preprocessed_df, config['columns']['target_col'])
X_oh, y_oh = electricity_preprocessor.to_tensor(preprocessed_oh_df, config['columns']['target_col'])

In [57]:
# Split into train/test sets
from sklearn.model_selection import train_test_split
# First split into train+val (80%) and test (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Then split train+val into train (87.5% of 80% = 70%) and val (12.5% of 80% = 10%)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.125, random_state=42)

# First split into train+val (80%) and test (20%)
X_oh_train_val, X_oh_test, y_oh_train_val, y_oh_test = train_test_split(X_oh, y_oh, test_size=0.2, random_state=42)
# Then split train+val into train (87.5% of 80% = 70%) and val (12.5% of 80% = 10%)
X_oh_train, X_oh_val, y_oh_train, y_oh_val = train_test_split(X_oh_train_val, y_oh_train_val, test_size=0.125, random_state=42)


In [58]:
# Save preprocessor state and processed data
electricity_preprocessor.save_preprocessor(save_dir=os.path.join(processed_data_path, 'electricity'))
electricity_preprocessor.save_data(X_train, X_val, X_test, y_train, y_val, y_test,
                             save_dir=os.path.join(processed_data_path, 'electricity'))
# Save preprocessor state and processed data
electricity_preprocessor.save_preprocessor(save_dir=os.path.join(processed_data_path, 'electricity_oh'))
electricity_preprocessor.save_data(X_oh_train, X_oh_val, X_oh_test, y_oh_train, y_oh_val, y_oh_test,
                             save_dir=os.path.join(processed_data_path, 'electricity_oh'))


In [59]:
# Verify saved data
loaded_X_train, loaded_X_val, loaded_X_test, loaded_y_train, loaded_y_val, loaded_y_test = \
    electricity_preprocessor.load_data(save_dir=os.path.join(processed_data_path, 'electricity'))
print(f"electricity - Loaded data shapes:")
print(f"X_train: {loaded_X_train.shape}, y_train: {loaded_y_train.shape}")

electricity - Loaded data shapes:
X_train: torch.Size([31717, 8]), y_train: torch.Size([31717])


### Covertype Dataset

In [60]:
config = CONFIG['dataset'][9]

In [61]:
covertype_preprocessor = preprocess.DatasetPreprocessor(numerical_cols=config['columns']['numerical_cols'] or [],
                                                    ordinal_cols=config['columns']['ordinal_cols'] or [],
                                                    binary_cols=config['columns']['binary_cols'] or [],
                                                    categorical_cols=config['columns']['categorical_cols'] or [],
                                                    target_col=config['columns']['target_col'])


In [62]:
# Load Covertype dataset
from scipy.io import arff
covertype_df = arff.loadarff(config['path'])
covertype_df = pd.DataFrame(covertype_df[0])
covertype_df.columns = config['columns']['all_cols']


In [63]:
# Preprocess dataset
preprocessed_df, preprocessed_oh_df, categories = covertype_preprocessor.preprocess(covertype_df)
print(f"Preprocessed DataFrame shape: {preprocessed_df.shape}")
print(f"Preprocessed One-Hot Encoded DataFrame shape: {preprocessed_oh_df.shape}")
print(f"Categories for embedding: {categories}")


Dropped columns with more than 50.0% missing values: []
Preprocessed DataFrame shape: (581012, 55)
Preprocessed One-Hot Encoded DataFrame shape: (581012, 55)
Categories for embedding: []


In [64]:
# Convert to tensors
X, y = covertype_preprocessor.to_tensor(preprocessed_df, config['columns']['target_col'])
X_oh, y_oh = covertype_preprocessor.to_tensor(preprocessed_oh_df, config['columns']['target_col'])


In [65]:
# Split into train/test sets
from sklearn.model_selection import train_test_split
# First split into train+val (80%) and test (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Then split train+val into train (87.5% of 80% = 70%) and val (12.5% of 80% = 10%)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.125, random_state=42)

# First split into train+val (80%) and test (20%)
X_oh_train_val, X_oh_test, y_oh_train_val, y_oh_test = train_test_split(X_oh, y_oh, test_size=0.2, random_state=42)
# Then split train+val into train (87.5% of 80% = 70%) and val (12.5% of 80% = 10%)
X_oh_train, X_oh_val, y_oh_train, y_oh_val = train_test_split(X_oh_train_val, y_oh_train_val, test_size=0.125, random_state=42)


In [66]:
# Save preprocessor state and processed data
covertype_preprocessor.save_preprocessor(save_dir=os.path.join(processed_data_path, 'covertype'))
covertype_preprocessor.save_data(X_train, X_val, X_test, y_train, y_val, y_test,
                             save_dir=os.path.join(processed_data_path, 'covertype'))
# Save preprocessor state and processed data
covertype_preprocessor.save_preprocessor(save_dir=os.path.join(processed_data_path, 'covertype_oh'))
covertype_preprocessor.save_data(X_oh_train, X_oh_val, X_oh_test, y_oh_train, y_oh_val, y_oh_test,
                             save_dir=os.path.join(processed_data_path, 'covertype_oh'))


In [67]:
# Verify saved data
loaded_X_train, loaded_X_val, loaded_X_test, loaded_y_train, loaded_y_val, loaded_y_test = \
    covertype_preprocessor.load_data(save_dir=os.path.join(processed_data_path, 'covertype'))
print(f"covertype - Loaded data shapes:")
print(f"X_train: {loaded_X_train.shape}, y_train: {loaded_y_train.shape}")



covertype - Loaded data shapes:
X_train: torch.Size([406707, 54]), y_train: torch.Size([406707])
